In [1]:
!git clone https://github.com/liuzhuang13/Transferable_RA.git

Cloning into 'Transferable_RA'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 24 (delta 6), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 745.53 KiB | 4.21 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
import sys
import torch

# 1. Point Python to the downloaded folder
sys.path.append('./Transferable_RA')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# 2. THE KAGGLE FILE PATH
# If you named your upload 'ra-model-weights', it will likely be here:
weights_path = '/kaggle/input/datasets/siddharthreddy1809/ra-model-weights/model_dn_ra_r18.pth' 

print("Loading the model...")
author_model = torch.load(weights_path, map_location=device, weights_only=False)

# Unwrap if necessary
if isinstance(author_model, torch.nn.DataParallel) or hasattr(author_model, 'module'):
    author_model = author_model.module

author_model = author_model.to(device)
author_model.eval()
print("model successfully loaded")

Running on: cuda
Loading the model...


/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'torch.nn.parallel.data_parallel.DataParallel' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'torch.nn.modules.conv.Conv2d' has changed. you can retrieve the original source code by accessing the object's source

model successfully loaded


/usr/local/lib/python3.12/dist-packages/torch/serialization.py:1639: SourceChangeWarning: source code of class 'torch.nn.modules.batchnorm.BatchNorm2d' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)


In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from PIL import Image
import torchvision.transforms as T

# --- 1. THE KAGGLE FILE PATHS ---
# Make sure these match the paths printed from your previous step!
IMAGES_FOLDER = '/kaggle/input/datasets/hsankesara/flickr-image-dataset/'
CSV_FILE = '/kaggle/input/flickr30k-images-with-captions/results.csv'

# --- 2. BUILD THE VOCABULARY ---
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def tokenize(self, text):
        return str(text).lower().split()

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] = frequencies.get(word, 0) + 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenize(text)
        return [self.stoi["<START>"]] + \
               [self.stoi.get(word, self.stoi["<UNK>"]) for word in tokenized_text] + \
               [self.stoi["<END>"]]

# --- 3. THE PYTORCH DATASET ---
class FlickrDataset(Dataset):
    def __init__(self, img_folder, csv_file, transform=None, freq_threshold=5):
        self.transform = transform
        self.img_folder = img_folder
        
        # Load the CSV. Flickr30k sometimes has a few corrupted lines, so we skip them!
        print("Reading CSV and cleaning data...")
        self.df = pd.read_csv(csv_file, delimiter='|', skipinitialspace=True, on_bad_lines='skip')
        
        # Sometimes the column names have extra spaces, let's strip them
        self.df.columns = self.df.columns.str.strip()
        
        # Drop any rows where the comment is missing
        self.df = self.df.dropna(subset=['comment'])
        
        self.imgs = self.df['image_name'].tolist()
        self.captions = self.df['comment'].tolist()

        # Initialize and build vocabulary
        print("Building the English Vocabulary...")
        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocabulary(self.captions)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]
        
        # Load the image
        img_path = os.path.join(self.img_folder, img_id.strip())
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        # Convert the sentence to numbers
        numericalized_caption = self.vocab.numericalize(caption)

        return img, torch.tensor(numericalized_caption)

# --- 4. THE BATCH COLLATOR (PADDING) ---
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        # Separate images and captions
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        
        targets = [item[1] for item in batch]
        # Pad the sentences so they all match the longest one in the batch!
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets

# --- 5. INITIALIZE EVERYTHING ---
print("Setting up transforms and DataLoader...")
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

# Create the dataset
dataset = FlickrDataset(
    img_folder=IMAGES_FOLDER,
    csv_file=CSV_FILE,
    transform=transform
)

pad_idx = dataset.vocab.stoi["<PAD>"]

# BATCH SIZE OF 32: A safe number so we don't blow up Kaggle's memory!
data_loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    num_workers=2, # Kaggle allows 2 CPU workers to load images faster
    shuffle=True,
    collate_fn=MyCollate(pad_idx=pad_idx)
)

print(f"DataLoader Ready! Total images to process: {len(dataset)}")
print(f"New Vocabulary Size: {len(dataset.vocab)} words")